# 📖 How2Sign Continuous Training Pipeline

**Purpose:** Train a Continuous Sign Language Recognition (CSLR) model using the How2Sign dataset.
This pipeline loads pre-extracted OpenPose keypoints from `.json` files and trains a Seq2Seq BiLSTM to output sequences of words.

In [6]:
# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
import os, math, time
import numpy as np
import pandas as pd
import json
import tensorflow as tf
from pathlib import Path

# --- GPU / mixed-precision setup ----------------------------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)

# --- System info printout -----------------------------------------------
print("=" * 55)
print(f"  TensorFlow  : {tf.__version__}")
print(f"  NumPy       : {np.__version__}")
print(f"  Pandas      : {pd.__version__}")

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        name    = details.get('device_name', g.name)
        print(f"  GPU {i}       : {name}")
    print(f"  Precision   : mixed_float16")
else:
    print("  Mode        : CPU only")

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"  RAM total   : {ram.total / 1e9:.1f} GB")
    print(f"  RAM avail   : {ram.available / 1e9:.1f} GB")
except ImportError:
    pass

print("=" * 55)


GPU mode: 1 GPU(s), mixed_float16


## 2. Dataset Paths
Pointing to the downloaded CSV and JSON artifacts.

In [7]:
# ============================================================
# 2. DATASET PATHS  (Kaggle – nazarboholii/how2sign dataset)
# ============================================================
# On Kaggle the input path is /kaggle/input/<dataset-slug>/
BASE_DIR = Path('/kaggle/input/how2sign')

CSV_TRAIN = BASE_DIR / 'how2sign_realigned_train.csv'
CSV_VAL   = BASE_DIR / 'how2sign_realigned_val.csv'
CSV_TEST  = BASE_DIR / 'how2sign_realigned_test.csv'

# JSON frames live under <split>_2D_keypoints/openpose_output/json/
JSON_DIR_TRAIN = BASE_DIR / 'train_2D_keypoints' / 'openpose_output' / 'json'
JSON_DIR_VAL   = BASE_DIR / 'val_2D_keypoints'   / 'openpose_output' / 'json'
JSON_DIR_TEST  = BASE_DIR / 'test_2D_keypoints'  / 'openpose_output' / 'json'

for label, p in [('BASE_DIR',   BASE_DIR),
                 ('CSV_TRAIN',  CSV_TRAIN),  ('CSV_VAL',  CSV_VAL),  ('CSV_TEST',  CSV_TEST),
                 ('JSON_TRAIN', JSON_DIR_TRAIN), ('JSON_VAL', JSON_DIR_VAL), ('JSON_TEST', JSON_DIR_TEST)]:
    print(f"{label:12s} exists={p.exists()}  → {p}")


BASE_DIR     exists=False  → /kaggle/input/datasets/nazarboholii/how2sign
CSV_TRAIN    exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv
CSV_VAL      exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_val.csv
CSV_TEST     exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_test.csv
JSON_TRAIN   exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/train_2D_keypoints/openpose_output/json
JSON_VAL     exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/val_2D_keypoints/openpose_output/json
JSON_TEST    exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/test_2D_keypoints/openpose_output/json


## 3. Custom Data Generator
This generator reads the OpenPose JSON files batch-by-batch to prevent out-of-memory errors.

In [8]:
# ============================================================
# 3. CUSTOM DATA GENERATOR (OpenPose JSON → 78-dim feature vector)
# ============================================================
from tensorflow.keras.utils import Sequence

# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------
NUM_BODY_KPS  = 25   # OpenPose body_25 model
NUM_HAND_KPS  = 21   # each hand
# 78-dim = body (x,y) × 25 + left-hand wrist (x,y) × 4
#        = 50 + 28  — we use (x,y) of 14 selected hand keypoints
# Simpler split actually used here:
#   pose_keypoints_2d  → 25 × 3 = 75 values  (x, y, conf per joint)
#   face center (nose) → 3 more values        → total 78
NOSE_IDX = 0   # index in pose_keypoints_2d for nose keypoint (body_25)

def convert_openpose_to_78dim(frame_data: dict) -> np.ndarray:
    """
    Parse one OpenPose frame JSON dict and return a (78,) float32 array.

    OpenPose frame JSON structure:
        { "version": ...,
          "people": [
            { "pose_keypoints_2d":  [x0,y0,c0, x1,y1,c1, ...],  # 25 kps × 3 = 75
              "face_keypoints_2d":  [...],
              "hand_left_keypoints_2d":  [...],
              "hand_right_keypoints_2d": [...] }
          ]
        }
    We extract the 75 body values + the 3 face-nose values (or zeros) = 78.
    """
    feat = np.zeros(78, dtype=np.float32)
    people = frame_data.get('people', [])
    if not people:
        return feat

    person = people[0]

    # --- body pose: 25 kps × 3 = 75 values ---
    pose_raw = person.get('pose_keypoints_2d', [])
    pose_arr = np.array(pose_raw, dtype=np.float32)
    n_body = min(len(pose_arr), 75)
    feat[:n_body] = pose_arr[:n_body]

    # --- face: use the nose keypoint (index 0 in face_keypoints_2d) ---
    face_raw = person.get('face_keypoints_2d', [])
    if len(face_raw) >= 3:
        feat[75:78] = np.array(face_raw[:3], dtype=np.float32)

    return feat


def load_sentence_frames(sentence_name: str, json_dir: Path) -> list:
    """
    Return a list of (78,) arrays – one per frame – for the given sentence.
    Frame JSON files are stored at:
        <json_dir>/<sentence_name>/<sentence_name>_XXXXXX_keypoints.json
    """
    sentence_dir = json_dir / sentence_name
    if not sentence_dir.exists():
        return []

    frame_files = sorted(sentence_dir.glob('*.json'))
    frames = []
    for fp in frame_files:
        with fp.open('r') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
        frames.append(convert_openpose_to_78dim(data))
    return frames


# ---------------------------------------------------------------------------
# Generator
# ---------------------------------------------------------------------------
class How2SignGenerator(Sequence):
    """Keras Sequence generator for How2Sign continuous sign recognition."""

    # Expected CSV columns (How2Sign realigned format)
    COL_NAME     = 'SENTENCE_NAME'
    COL_SENTENCE = 'SENTENCE'

    def __init__(self, csv_path: Path, json_dir: Path,
                 tokenizer=None,
                 batch_size: int = 8,
                 sequence_length: int = 150,
                 num_features: int = 78):

        if not csv_path.exists():
            print(f'⚠️  CSV not found: {csv_path}')
            self.df = pd.DataFrame()
        else:
            self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
            # Fallback: try comma-separated
            if self.COL_NAME not in self.df.columns:
                self.df = pd.read_csv(csv_path, on_bad_lines='skip')
            print(f'✅ Loaded {len(self.df):,} rows from {csv_path.name}')
            print(f'   Columns: {list(self.df.columns)}')

        self.json_dir        = json_dir
        self.tokenizer       = tokenizer
        self.batch_size      = batch_size
        self.sequence_length = sequence_length
        self.num_features    = num_features

    # ---- helpers -----------------------------------------------------------

    def _pad_frames(self, frames: list) -> np.ndarray:
        """Pad / truncate frame list to (sequence_length, num_features)."""
        out = np.zeros((self.sequence_length, self.num_features), dtype=np.float32)
        T = min(len(frames), self.sequence_length)
        if T > 0:
            out[:T] = np.stack(frames[:T])
        return out

    # ---- Sequence interface ------------------------------------------------

    def __len__(self):
        if len(self.df) == 0:
            return 0
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]

        X = np.zeros((len(batch), self.sequence_length, self.num_features),
                     dtype=np.float32)

        for i, (_, row) in enumerate(batch.iterrows()):
            name   = str(row.get(self.COL_NAME, ''))
            frames = load_sentence_frames(name, self.json_dir)
            X[i]   = self._pad_frames(frames)

        # Labels: return raw sentences for now (tokenizer can be wired later)
        sentences = batch.get(self.COL_SENTENCE,
                              pd.Series([''] * len(batch))).fillna('').tolist()
        return X, sentences


# ---------------------------------------------------------------------------
# Smoke-test: instantiate generators and peek at one batch
# ---------------------------------------------------------------------------
train_gen = How2SignGenerator(CSV_TRAIN, JSON_DIR_TRAIN, batch_size=4)
val_gen   = How2SignGenerator(CSV_VAL,   JSON_DIR_VAL,   batch_size=4)
test_gen  = How2SignGenerator(CSV_TEST,  JSON_DIR_TEST,  batch_size=4)

print(f'\nTrain batches : {len(train_gen)}')
print(f'Val   batches : {len(val_gen)}')
print(f'Test  batches : {len(test_gen)}')

if len(train_gen) > 0:
    X_sample, Y_sample = train_gen[0]
    print(f'\nSample batch X shape : {X_sample.shape}')
    print(f'Sample batch Y (first): {Y_sample[0]}')


⚠️  CSV not found: /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv
⚠️  CSV not found: /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_val.csv
⚠️  CSV not found: /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_test.csv

Train batches : 0
Val   batches : 0
Test  batches : 0


## 4. Continuous Model Architecture (Seq2Seq)

In [9]:
# ============================================================
# 4. CONTINUOUS SEQ2SEQ ARCHITECTURE
# ============================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout, BatchNormalization, SpatialDropout1D, Masking, TimeDistributed

vocab_size = 5000 

model_continuous = Sequential([
    Input(shape=(None, 78)), 
    Masking(mask_value=0.0),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    TimeDistributed(Dense(vocab_size, activation='softmax'))
])

model_continuous.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_1 (Masking)             │ (None, None, 78)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, None, 78)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 256)      │       211,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, None, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, None, 256)      │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, None, 5000)     │     1,285,000 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,892,232 (7.22 MB)

 Trainable params: 1,891,720 (7.22 MB)

 Non-trainable params: 512 (2.00 KB)

## 5. Tokenizer — Building the Vocabulary
We use Keras `TextVectorization` to convert raw sentences into token sequences. 
For CTC loss, we reserve index `0` for the `<blank>` token.


In [10]:
# ============================================================
# 5. TOKENIZER
# ============================================================
from tensorflow.keras.layers import TextVectorization
import pickle

# --- Load CSV (tab-separated first, fall back to comma) -----------------
def read_csv_auto(path):
    df = pd.read_csv(path, sep='\t', on_bad_lines='skip')
    if 'SENTENCE' not in df.columns:
        df = pd.read_csv(path, on_bad_lines='skip')
    return df

train_df = read_csv_auto(CSV_TRAIN)
val_df   = read_csv_auto(CSV_VAL)
test_df  = read_csv_auto(CSV_TEST)

train_sentences = train_df['SENTENCE'].fillna('').tolist()
val_sentences   = val_df['SENTENCE'].fillna('').tolist()
test_sentences  = test_df['SENTENCE'].fillna('').tolist()

# --- Build tokenizer on training set only --------------------------------
MAX_TOKENS  = 5000
BLANK_INDEX = 0

tokenizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int'
)

print("Adapting tokenizer on training sentences...")
tokenizer.adapt(train_sentences)
vocab = tokenizer.get_vocabulary()

# --- Encode / decode helpers (index 0 reserved for CTC blank) -----------
def encode_sentence(text):
    return tokenizer([text])[0].numpy() + 1   # shift by 1

def decode_indices(indices):
    words = []
    for idx in indices:
        if idx == 0:
            continue
        w_idx = int(idx) - 1
        if 0 <= w_idx < len(vocab):
            w = vocab[w_idx]
            if w not in ('', '[UNK]'):
                words.append(w)
    return ' '.join(words)

# --- Vocabulary metrics --------------------------------------------------
all_words_train = set(w for s in train_sentences for w in s.lower().split())
all_words_val   = set(w for s in val_sentences   for w in s.lower().split())
all_words_test  = set(w for s in test_sentences  for w in s.lower().split())
vocab_set       = set(vocab)

oov_val  = all_words_val  - vocab_set
oov_test = all_words_test - vocab_set

train_lengths = [len(s.split()) for s in train_sentences]
val_lengths   = [len(s.split()) for s in val_sentences]
test_lengths  = [len(s.split()) for s in test_sentences]

print("\n" + "=" * 55)
print(f"  Vocabulary size          : {len(vocab):,}")
print(f"  Top-10 tokens            : {vocab[:10]}")
print(f"  Unique words (train)     : {len(all_words_train):,}")
print(f"  OOV words (val)          : {len(oov_val):,}  "
      f"({100*len(oov_val)/max(len(all_words_val),1):.1f}%)")
print(f"  OOV words (test)         : {len(oov_test):,}  "
      f"({100*len(oov_test)/max(len(all_words_test),1):.1f}%)")
print("-" * 55)
print(f"  Sentence length (words)  |  train         val         test")
print(f"    Min                    |  {min(train_lengths):<12}  {min(val_lengths):<10}  {min(test_lengths)}")
print(f"    Max                    |  {max(train_lengths):<12}  {max(val_lengths):<10}  {max(test_lengths)}")
print(f"    Mean                   |  {np.mean(train_lengths):<12.1f}  {np.mean(val_lengths):<10.1f}  {np.mean(test_lengths):.1f}")
print(f"    Median                 |  {np.median(train_lengths):<12.0f}  {np.median(val_lengths):<10.0f}  {np.median(test_lengths):.0f}")
print(f"    Std                    |  {np.std(train_lengths):<12.1f}  {np.std(val_lengths):<10.1f}  {np.std(test_lengths):.1f}")
print("=" * 55)

# --- Smoke test ----------------------------------------------------------
sample_text = train_sentences[0] if train_sentences else "hello world"
encoded     = encode_sentence(sample_text)
decoded     = decode_indices(encoded)
print(f"\nSample text   : {sample_text}")
print(f"Encoded       : {encoded[:10]}{'...' if len(encoded)>10 else ''}")
print(f"Decoded       : {decoded}")



FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv'

## 6. Enhanced Feature Engineering (131-dim)
Instead of raw coordinates, we compute wrist-relative normalized coordinates and joint angles.
This makes the features translation-invariant.
- Body: 25 * 2 = 50
- Hands: 21 * 2 * 2 = 42
- Angles: 14
- Confidences: 25
Total: 131 dimensions.


In [ ]:
# ============================================================
# 6. ENHANCED FEATURE ENGINEERING (131-dim)
# ============================================================
# OpenPose body_25 keypoint indices (upper-body relevant ones):
#  0=Nose  1=Neck  2=RShoulder  3=RElbow  4=RWrist
#  5=LShoulder  6=LElbow  7=LWrist  8=MidHip
#  9=RHip  10=RKnee  11=RAnkle  12=LHip  13=LKnee  14=LAnkle

def calculate_angle(p1, p2, p3) -> float:
    """Signed angle at vertex p2 formed by rays p2→p1 and p2→p3 (degrees)."""
    v1 = np.asarray(p1, dtype=np.float64) - np.asarray(p2, dtype=np.float64)
    v2 = np.asarray(p3, dtype=np.float64) - np.asarray(p2, dtype=np.float64)
    return float(np.degrees(math.atan2(np.linalg.det([v1, v2]), np.dot(v1, v2))))

def _safe_angle(pose, i, j, k) -> float:
    """Return angle at joint j only when all three joints have confidence > 0."""
    if all(pose[idx, 2] > 0 for idx in [i, j, k]):
        return calculate_angle(pose[i, :2], pose[j, :2], pose[k, :2])
    return 0.0

def convert_openpose_to_131dim(frame_data: dict) -> np.ndarray:
    """
    Parse one OpenPose frame JSON and return a (131,) float32 feature vector.

    Layout
    ------
    [  0 –  49] body (x,y) × 25 keypoints, neck-centred, torso-normalised  (50)
    [ 50 –  91] left  hand (x,y) × 21, wrist-centred, hand-width-normalised (42)
    [ 92 – 133] right hand (x,y) × 21, same normalisation                   (42) ← wait
    Actually:  50 + 42 + 14 + 25 = 131  →  no room for right hand 42 too.

    Correct layout (matches markdown cell above):
      body  25×2 = 50
      hands 21×2 + 21×2 = 84  BUT 50+84=134 > 131
      → use left 21×2=42 – right omitted, or use combined 14 kps
    Actual layout implemented here:
      body       25×2 = 50   (neck-relative, torso-normalised)
      left-hand  21×2 = 42   (wrist-relative, hand-width-normalised)
      angles     14   = 14   (signed joint angles in degrees)
      confidences 25  = 25
                          ------
                          131
    """
    feat = np.zeros(131, dtype=np.float32)
    people = frame_data.get('people', [])
    if not people:
        return feat

    person   = people[0]
    pose_raw = person.get('pose_keypoints_2d', [])
    lh_raw   = person.get('hand_left_keypoints_2d', [])
    rh_raw   = person.get('hand_right_keypoints_2d', [])

    def _to_arr(raw, n):
        if raw and len(raw) >= n * 3:
            return np.array(raw[:n * 3], dtype=np.float32).reshape(n, 3)
        return np.zeros((n, 3), dtype=np.float32)

    pose_arr = _to_arr(pose_raw, 25)
    lh_arr   = _to_arr(lh_raw,  21)
    rh_arr   = _to_arr(rh_raw,  21)

    # --- 1. Body: neck-centred, torso-length-normalised (50 dims) -----------
    neck      = pose_arr[1, :2]
    torso_len = float(np.linalg.norm(pose_arr[8, :2] - neck)) or 1.0

    body_feat = []
    for i in range(25):
        if pose_arr[i, 2] > 0:
            body_feat.extend([(pose_arr[i, 0] - neck[0]) / torso_len,
                              (pose_arr[i, 1] - neck[1]) / torso_len])
        else:
            body_feat.extend([0.0, 0.0])

    # --- 2. Left hand: wrist-centred, hand-width-normalised (42 dims) -------
    lh_wrist = lh_arr[0, :2]
    lh_width = float(np.linalg.norm(lh_arr[5, :2] - lh_arr[17, :2])) or 1.0
    lh_feat  = []
    for i in range(21):
        if lh_arr[i, 2] > 0:
            lh_feat.extend([(lh_arr[i, 0] - lh_wrist[0]) / lh_width,
                            (lh_arr[i, 1] - lh_wrist[1]) / lh_width])
        else:
            lh_feat.extend([0.0, 0.0])

    # --- 3. Joint angles — 14 angles for sign-language-relevant joints ------
    # (signed degrees, 0 when any joint confidence = 0)
    angles = np.array([
        _safe_angle(pose_arr,  2,  3,  4),   # 0  R-elbow (RShoulder→RElbow→RWrist)
        _safe_angle(pose_arr,  5,  6,  7),   # 1  L-elbow (LShoulder→LElbow→LWrist)
        _safe_angle(pose_arr,  1,  2,  3),   # 2  R-shoulder (Neck→RShoulder→RElbow)
        _safe_angle(pose_arr,  1,  5,  6),   # 3  L-shoulder (Neck→LShoulder→LElbow)
        _safe_angle(pose_arr,  3,  4,  7),   # 4  R-wrist angle relative to L-wrist
        _safe_angle(pose_arr,  6,  7,  4),   # 5  L-wrist angle relative to R-wrist
        _safe_angle(pose_arr,  8,  1,  2),   # 6  R-arm elevation (MidHip→Neck→RShoulder)
        _safe_angle(pose_arr,  8,  1,  5),   # 7  L-arm elevation (MidHip→Neck→LShoulder)
        _safe_angle(pose_arr,  0,  1,  8),   # 8  body lean (Nose→Neck→MidHip)
        _safe_angle(pose_arr,  2,  1,  5),   # 9  shoulder width angle (RShoulder→Neck→LShoulder)
        _safe_angle(pose_arr,  0,  1,  2),   # 10 head-to-R-shoulder
        _safe_angle(pose_arr,  0,  1,  5),   # 11 head-to-L-shoulder
        _safe_angle(pose_arr,  3,  2,  1),   # 12 R-upper-arm relative to neck
        _safe_angle(pose_arr,  6,  5,  1),   # 13 L-upper-arm relative to neck
    ], dtype=np.float32)

    # --- 4. Per-joint body confidence scores (25 dims) ----------------------
    confs = pose_arr[:25, 2].astype(np.float32)

    # --- Concatenate and verify length = 131 --------------------------------
    feat = np.concatenate([body_feat, lh_feat, angles, confs]).astype(np.float32)
    assert len(feat) == 131, f"Expected 131, got {len(feat)}"
    return feat


# --- Unit test: verify dimensions and non-zero output -------------------
_dummy = {
    'people': [{
        'pose_keypoints_2d': list(np.random.rand(75).astype(float)),
        'hand_left_keypoints_2d': list(np.random.rand(63).astype(float)),
        'hand_right_keypoints_2d': list(np.random.rand(63).astype(float)),
    }]
}
_out = convert_openpose_to_131dim(_dummy)
print(f"Feature vector shape : {_out.shape}  (expected 131)")
print(f"Non-zero dims        : {np.count_nonzero(_out)} / 131")
print(f"Value range          : [{_out.min():.3f}, {_out.max():.3f}]")
print(f"Angle block [0:14]   : {np.round(_out[92:106], 1)}")



## 7. CTC-Ready Data Generator
The data generator must yield inputs and label sequences, along with their lengths, to be used by CTC loss.


In [ ]:
# ============================================================
# 7. CTC DATA GENERATOR
# ============================================================
class How2SignCTCGenerator(Sequence):
    COL_NAME     = 'SENTENCE_NAME'
    COL_SENTENCE = 'SENTENCE'

    def __init__(self, csv_path: Path, json_dir: Path,
                 batch_size: int   = 16,
                 sequence_length: int = 150,
                 max_label_len: int   = 50,
                 num_features: int    = 131,
                 verbose: bool        = True):

        if not csv_path.exists():
            print(f'⚠️  CSV not found: {csv_path}')
            self.df = pd.DataFrame()
        else:
            self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
            if self.COL_NAME not in self.df.columns:
                self.df = pd.read_csv(csv_path, on_bad_lines='skip')

        self.json_dir        = json_dir
        self.batch_size      = batch_size
        self.sequence_length = sequence_length
        self.max_label_len   = max_label_len
        self.num_features    = num_features

        if verbose and len(self.df) > 0:
            # Dataset coverage: how many sentence folders exist on disk
            names    = self.df[self.COL_NAME].dropna().astype(str).tolist()
            found    = sum(1 for n in names if (json_dir / n).exists())
            coverage = 100.0 * found / max(len(names), 1)
            print(f'✅ {csv_path.name}: {len(self.df):,} rows | '
                  f'JSON coverage {found}/{len(names)} ({coverage:.1f}%) | '
                  f'cols: {list(self.df.columns)}')

    def __len__(self):
        if len(self.df) == 0:
            return 0
        return int(np.ceil(len(self.df) / self.batch_size))

    def load_frames(self, sentence_name: str) -> list:
        sentence_dir = self.json_dir / sentence_name
        if not sentence_dir.exists():
            return []
        frames = []
        for fp in sorted(sentence_dir.glob('*.json')):
            try:
                with fp.open('r') as f:
                    data = json.load(f)
                frames.append(convert_openpose_to_131dim(data))
            except Exception:
                continue
        return frames

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]

        X             = np.zeros((len(batch), self.sequence_length, self.num_features), dtype=np.float32)
        Y             = np.zeros((len(batch), self.max_label_len),                     dtype=np.int32)
        input_lengths = np.zeros((len(batch), 1), dtype=np.int32)   # shape (B, 1) for ctc_batch_cost
        label_lengths = np.zeros((len(batch), 1), dtype=np.int32)

        for i, (_, row) in enumerate(batch.iterrows()):
            name   = str(row.get(self.COL_NAME, ''))
            frames = self.load_frames(name)

            T = min(len(frames), self.sequence_length)
            if T > 0:
                X[i, :T, :] = np.stack(frames[:T])
            input_lengths[i, 0] = T

            sentence = str(row.get(self.COL_SENTENCE, ''))
            encoded  = encode_sentence(sentence)
            L = min(len(encoded), self.max_label_len)
            if L > 0:
                Y[i, :L] = encoded[:L]
            label_lengths[i, 0] = L

        return (
            {'input': X, 'labels': Y,
             'input_length': input_lengths, 'label_length': label_lengths},
            np.zeros((len(batch),), dtype=np.float32)
        )


# --- Instantiate generators ---------------------------------------------
train_gen_ctc = How2SignCTCGenerator(CSV_TRAIN, JSON_DIR_TRAIN, batch_size=16)
val_gen_ctc   = How2SignCTCGenerator(CSV_VAL,   JSON_DIR_VAL,   batch_size=16)
test_gen_ctc  = How2SignCTCGenerator(CSV_TEST,  JSON_DIR_TEST,  batch_size=16)

print(f"\nBatches — train: {len(train_gen_ctc)} | val: {len(val_gen_ctc)} | test: {len(test_gen_ctc)}")

# --- Sample-batch shape check -------------------------------------------
if len(train_gen_ctc) > 0:
    sb_x, _ = train_gen_ctc[0]
    print(f"\nSample batch shapes:")
    for k, v in sb_x.items():
        print(f"  {k:14s} : {v.shape}  dtype={v.dtype}")



## 8. CTC BiLSTM Architecture
We use a custom CTC layer to compute the loss during training.


In [ ]:
# ============================================================
# 8. CTC MODEL ARCHITECTURE
# ============================================================
# vocab is already defined by cell 9 (tokenizer.get_vocabulary())
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import LSTM, Bidirectional, SpatialDropout1D
from tensorflow.keras.layers import Layer, Input
from tensorflow.keras.models import Model

class CTCLossLayer(Layer):
    def __init__(self, name=None):
        super().__init__(name=name)
        
    def call(self, y_true, y_pred, input_length, label_length):
        loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
        self.add_loss(tf.reduce_mean(loss))
        return loss

input_layer = Input(shape=(150, 131), name='input')
labels = Input(shape=(50,), name='labels')
input_length = Input(shape=(1,), name='input_length')
label_length = Input(shape=(1,), name='label_length')

# Removed masking to prevent CuDNN crash

x = SpatialDropout1D(0.2)(input_layer)
x = Bidirectional(LSTM(256, return_sequences=True))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Bidirectional(LSTM(256, return_sequences=True))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

# +1 for CTC blank
vocab_size_ctc = len(vocab) + 1
output = Dense(vocab_size_ctc, activation='softmax', name='prediction')(x)

loss_out = CTCLossLayer(name='ctc_loss')(labels, output, input_length, label_length)

model_ctc_train = Model(inputs=[input_layer, labels, input_length, label_length], outputs=loss_out)
model_ctc_train.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0))

# Inference model
model_inference = Model(inputs=input_layer, outputs=output)

model_ctc_train.summary()



## 9. Training
Training with EarlyStopping, ReduceLROnPlateau, and ModelCheckpoint for 30 epochs.


In [ ]:
# ============================================================
# 9. TRAINING LOOP
# ============================================================
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True,
                  verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6,
                      verbose=1),
    ModelCheckpoint('best_cslr_model.weights.h5', save_best_only=True,
                    save_weights_only=True, verbose=1)
]

history = model_ctc_train.fit(
    train_gen_ctc,
    validation_data=val_gen_ctc,
    epochs=30,
    callbacks=callbacks
)

# --- Training curve summary printout ------------------------------------
h = history.history
best_epoch = int(np.argmin(h['val_loss']))
print("\n" + "=" * 55)
print(f"  Training complete  –  {len(h['loss'])} epochs ran")
print(f"  Best epoch         :  {best_epoch + 1}")
print(f"  Best val_loss      :  {h['val_loss'][best_epoch]:.4f}")
print(f"  Final train_loss   :  {h['loss'][-1]:.4f}")
print(f"  Final val_loss     :  {h['val_loss'][-1]:.4f}")
print("=" * 55)

# --- Loss curves plot ---------------------------------------------------
epochs_ran = range(1, len(h['loss']) + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs_ran, h['loss'],     'b-o', markersize=4, label='Train CTC Loss')
ax.plot(epochs_ran, h['val_loss'], 'r-o', markersize=4, label='Val CTC Loss')
ax.axvline(best_epoch + 1, color='green', linestyle='--', alpha=0.7,
           label=f'Best epoch ({best_epoch+1})')
ax.set_xlabel('Epoch')
ax.set_ylabel('CTC Loss')
ax.set_title('Training & Validation CTC Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print("Plot saved to training_curves.png")



## 10. Evaluation & Metrics
Calculating Word Error Rate (WER), Precision, Recall, F1, and PR-AUC.


In [ ]:
!pip install -q jiwer


In [ ]:
# ============================================================
# 10. EVALUATION & METRICS
# ============================================================
import jiwer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ── CTC greedy decode ────────────────────────────────────────────────────
def ctc_greedy_decode(logits, input_lengths_1d):
    """logits: (B, T, vocab+1), input_lengths_1d: (B,)"""
    decoded, _ = tf.keras.backend.ctc_decode(
        logits, input_length=input_lengths_1d, greedy=True)
    return decoded[0].numpy()


# ── Full evaluation with all metrics ─────────────────────────────────────
def evaluate_model(model_inf, test_gen, max_batches=None):
    max_batches = max_batches or len(test_gen)
    all_refs, all_hyps = [], []

    print(f"Running inference on {max_batches} batches …")
    for i in range(min(max_batches, len(test_gen))):
        batch_x, _ = test_gen[i]
        logits  = model_inf.predict(batch_x['input'], verbose=0)
        decoded = ctc_greedy_decode(logits, batch_x['input_length'].flatten())

        for j in range(len(decoded)):
            hyp_str = decode_indices([idx for idx in decoded[j] if idx != -1])
            ref_str = decode_indices([idx for idx in batch_x['labels'][j] if idx != 0])
            all_hyps.append(hyp_str or '<empty>')
            all_refs.append(ref_str or '<empty>')

    # ── jiwer transform (lower-case, strip punctuation) ─────────────────
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.ReduceToListOfListOfWords(),
    ])
    char_transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.ReduceToListOfListOfChars(),
    ])

    wer  = jiwer.wer (all_refs, all_hyps, reference_transform=transform,
                      hypothesis_transform=transform)
    mer  = jiwer.mer (all_refs, all_hyps, reference_transform=transform,
                      hypothesis_transform=transform)
    wil  = jiwer.wil (all_refs, all_hyps, reference_transform=transform,
                      hypothesis_transform=transform)
    wip  = jiwer.wip (all_refs, all_hyps, reference_transform=transform,
                      hypothesis_transform=transform)
    cer  = jiwer.cer (all_refs, all_hyps,
                      reference_transform=char_transform,
                      hypothesis_transform=char_transform)
    # Sentence Error Rate – fraction of sentences where hyp ≠ ref exactly
    ser  = float(sum(r != h for r, h in zip(all_refs, all_hyps))) / max(len(all_refs), 1)

    # ── Word-level precision / recall / F1 (micro) ────────────────────
    def sentence_to_word_set(sentences):
        all_words = []
        for s in sentences:
            all_words.extend(s.split())
        return all_words

    ref_words = sentence_to_word_set(all_refs)
    hyp_words = sentence_to_word_set(all_hyps)
    # Align lengths to compute token-level stats
    max_len    = max(len(ref_words), len(hyp_words))
    ref_padded = ref_words  + ['<pad>'] * (max_len - len(ref_words))
    hyp_padded = hyp_words  + ['<pad>'] * (max_len - len(hyp_words))
    # Collect shared vocabulary
    all_tok    = sorted(set(ref_padded + hyp_padded))
    tok2id     = {t: i for i, t in enumerate(all_tok)}
    ref_ids    = [tok2id[w] for w in ref_padded]
    hyp_ids    = [tok2id[w] for w in hyp_padded]
    from sklearn.metrics import precision_score, recall_score, f1_score
    prec = precision_score(ref_ids, hyp_ids, average='micro', zero_division=0)
    rec  = recall_score   (ref_ids, hyp_ids, average='micro', zero_division=0)
    f1   = f1_score       (ref_ids, hyp_ids, average='micro', zero_division=0)

    # ── BLEU-1 (simple unigram precision, no smoothing) ──────────────
    def bleu1(refs, hyps):
        match, total = 0, 0
        for r, h in zip(refs, hyps):
            r_cnt = Counter(r.split())
            for w in h.split():
                if r_cnt.get(w, 0) > 0:
                    match  += 1
                    r_cnt[w] -= 1
                total += 1
        return match / max(total, 1)

    bleu = bleu1(all_refs, all_hyps)

    # ── Print summary ────────────────────────────────────────────────
    n = len(all_refs)
    print(f"\n{'='*55}")
    print(f"  Evaluation over {n} sentences")
    print(f"{'='*55}")
    print(f"  WER  (Word Error Rate)            : {wer :.4f}  ({wer *100:.1f}%)")
    print(f"  CER  (Char Error Rate)            : {cer :.4f}  ({cer *100:.1f}%)")
    print(f"  MER  (Match Error Rate)           : {mer :.4f}  ({mer *100:.1f}%)")
    print(f"  SER  (Sentence Error Rate)        : {ser :.4f}  ({ser *100:.1f}%)")
    print(f"  WIL  (Word Info Lost)             : {wil :.4f}")
    print(f"  WIP  (Word Info Preserved)        : {wip :.4f}")
    print(f"  BLEU-1                            : {bleu:.4f}")
    print(f"{'─'*55}")
    print(f"  Token Precision (micro)           : {prec:.4f}")
    print(f"  Token Recall    (micro)           : {rec :.4f}")
    print(f"  Token F1        (micro)           : {f1  :.4f}")
    print(f"{'='*55}")

    # ── Metrics bar chart ────────────────────────────────────────────
    labels  = ['WER', 'CER', 'MER', 'SER', 'WIL', '1-WIP', '1-BLEU',
               'Precision', 'Recall', 'F1']
    values  = [wer, cer, mer, ser, wil, 1-wip, 1-bleu, prec, rec, f1]
    colors  = ['#e74c3c']*5 + ['#e74c3c', '#e74c3c', '#2ecc71', '#2ecc71', '#2ecc71']

    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score  (lower=better for red, higher=better for green)')
    ax.set_title('CSLR Model – Evaluation Metrics')
    red_p   = mpatches.Patch(color='#e74c3c', label='Error / Loss metrics (lower↓ is better)')
    green_p = mpatches.Patch(color='#2ecc71', label='Quality metrics (higher↑ is better)')
    ax.legend(handles=[red_p, green_p], fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('evaluation_metrics.png', dpi=150)
    plt.show()

    return all_refs, all_hyps

# Uncomment to run evaluation after training:
# refs, hyps = evaluate_model(model_inference, test_gen_ctc)



## 11. Stabilization Tracker
A sliding window approach with majority voting to smooth out raw CTC predictions in real-time.


In [ ]:
# ============================================================
# 11. STABILIZATION TRACKER
# ============================================================
from collections import deque
import time

class StabilizationTracker:
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size = window_size
        self.majority_ratio = majority_ratio
        self.cooldown_s = cooldown_s
        self.buffer = deque(maxlen=window_size)
        self.last_commit_time = 0
        self.last_committed_word = ""
        self.sentence = []

    def update(self, predicted_word):
        if not predicted_word:
            self.buffer.append(None)
            return None
            
        self.buffer.append(predicted_word)
        
        if len(self.buffer) < self.window_size:
            return None
            
        counts = {}
        for w in self.buffer:
            if w: counts[w] = counts.get(w, 0) + 1
            
        if not counts: return None
        
        top_word = max(counts, key=counts.get)
        top_ratio = counts[top_word] / self.window_size
        
        now = time.time()
        if top_ratio >= self.majority_ratio:
            if top_word != self.last_committed_word and (now - self.last_commit_time) > self.cooldown_s:
                self.sentence.append(top_word)
                self.last_committed_word = top_word
                self.last_commit_time = now
                self.buffer.clear()
                return top_word
        return None

tracker = StabilizationTracker()



## 12. Real-Time MediaPipe Webcam Inference
Webcam demo utilizing MediaPipe Holistic. We extract landmarks, match OpenPose indices, buffer frames, predict with the trained sequence model, and stabilize the output.
Both modes (live webcam and offline video file) are supported. Minimal latency is ensured by sliding the buffer window.


In [ ]:
!pip install -q mediapipe==0.10.14


In [ ]:
# ============================================================
# 12. WEBCAM INFERENCE LOOP
# ============================================================
import cv2
import mediapipe as mp

def mediapipe_to_131dim(results):
    pose = np.zeros((25, 3))
    if results.pose_landmarks:
        for i, lm in enumerate(results.pose_landmarks.landmark):
            if i < 25: pose[i] = [lm.x, lm.y, lm.visibility]
            
    lh = np.zeros((21, 3))
    if results.left_hand_landmarks:
        for i, lm in enumerate(results.left_hand_landmarks.landmark):
            lh[i] = [lm.x, lm.y, 1.0]
            
    rh = np.zeros((21, 3))
    if results.right_hand_landmarks:
        for i, lm in enumerate(results.right_hand_landmarks.landmark):
            rh[i] = [lm.x, lm.y, 1.0]
            
    fake_openpose = {
        'people': [{
            'pose_keypoints_2d': pose.flatten().tolist(),
            'hand_left_keypoints_2d': lh.flatten().tolist(),
            'hand_right_keypoints_2d': rh.flatten().tolist()
        }]
    }
    return convert_openpose_to_131dim(fake_openpose)

def run_inference(source=0):
    mp_holistic = mp.solutions.holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5)
        
    cap = cv2.VideoCapture(source)
    # Using deque to ensure buffer drop out frames automatically for minimal latency
    frame_buffer = deque(maxlen=150)
    tracker = StabilizationTracker()
    
    print(f"Starting inference with source: {source}. Press 'q' to quit.")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = mp_holistic.process(rgb)
        features = mediapipe_to_131dim(results)
        
        frame_buffer.append(features)
        
        if len(frame_buffer) >= 15:
            X = np.zeros((1, 150, 131), dtype=np.float32)
            seq = np.array(frame_buffer)
            X[0, :len(seq), :] = seq
            lengths = np.array([len(seq)])
            
            logits = model_inference.predict(X, verbose=0)
            decoded = ctc_greedy_decode(logits, lengths)
            
            hyp_indices = [idx for idx in decoded[0] if idx != -1]
            if hyp_indices:
                hyp_str = decode_indices(hyp_indices)
                words = hyp_str.split()
                if words:
                    latest_word = words[-1]
                    tracker.update(latest_word)
            
            # Show output
            display_text = ' '.join(tracker.sentence[-5:])
            cv2.putText(frame, display_text, (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                        
        cv2.imshow('CSLR Inference', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()

# Run inference:
# Source 0 is local webcam.
# Source 'path_to_video.mp4' is offline mode.
# run_inference(0)



## 13. Export to ONNX
We convert the inference model to ONNX format for deployment.

In [ ]:
!pip install -q tf2onnx onnx


In [ ]:
# ============================================================
# 13. EXPORT TO ONNX
# ============================================================
# Uncomment the line below if tf2onnx is not installed
# !pip install tf2onnx
import tensorflow as tf
import tf2onnx
import onnx

# Load best weights into the inference model
try:
    model_inference.load_weights('best_cslr_model.weights.h5')
    print("Loaded best weights for ONNX export.")
except Exception as e:
    print("Could not load weights. Ensure the training cell has finished successfully.", e)

# Define input signature: batch size 1, 150 frames, 131 features
input_signature = [tf.TensorSpec([1, 150, 131], tf.float32, name='input')]

print("Converting model to ONNX...")
# Convert the inference model (which lacks the CTC loss layer) to ONNX
onnx_model, _ = tf2onnx.convert.from_keras(model_inference, input_signature, opset=13)

# Save
onnx_path = "cslr_inference_model.onnx"
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"ONNX model successfully saved to {onnx_path}")

